# UCS3564 – NLP | Assignment 5: Binary Text Classification using BERT

**Dataset:** Kaggle – *IMDB Dataset of 50K Movie Reviews* (`lakshmi25npathi/imdb-dataset-of-50k-movie-reviews`)
**Model:** `bert-base-uncased` + classification head, fine-tuned on the dataset.
**Evaluation:** accuracy, precision, recall, F1, confusion matrix (+ a TF-IDF/Logistic-Regression bag-of-words baseline trained on the same split).

> Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
!pip install -q transformers kagglehub pandas matplotlib scikit-learn nltk ipywidgets accelerate

In [ ]:
import os, re, glob, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import nltk
from IPython.display import display
from nltk.tokenize import WhitespaceTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertForSequenceClassification, BertTokenizerFast, get_linear_schedule_with_warmup

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_NAMES = ["Negative", "Positive"]

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)

In [ ]:
# ----------------------------- Configuration ------------------------------ #
MODEL_NAME  = "bert-base-uncased"
NUM_SAMPLES = None     # None = use all 50,000 reviews. Set e.g. 20000 for a faster run.
MAX_LEN     = 256
BATCH_SIZE  = 32
EPOCHS      = 2
LEARNING_RATE = 2e-5
SAVE_DIR    = "bert_imdb_model"

## 1. Load the Kaggle IMDB dataset
Downloaded via `kagglehub`; if that fails you will be asked to upload `IMDB Dataset.csv` (columns: `review`, `sentiment`).

In [ ]:
csv_path = None
try:
    import kagglehub
    dataset_path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
    matches = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
    csv_path = matches[0] if matches else None
    print("Downloaded to:", dataset_path)
except Exception as error:
    print("kagglehub download failed:", error)

if csv_path is None:
    try:
        from google.colab import files
        print("Please upload 'IMDB Dataset.csv' from Kaggle.")
        uploaded = files.upload()
        csv_path = [n for n in uploaded.keys() if n.endswith(".csv")][0]
    except Exception as error:
        raise RuntimeError("Place 'IMDB Dataset.csv' in the working directory and set csv_path.") from error

raw_df = pd.read_csv(csv_path)
print("Using:", csv_path, "| shape:", raw_df.shape)
display(raw_df.head())
print(raw_df["sentiment"].value_counts())

## 2. Text cleaning (NLTK) and train / validation / test split
* Remove HTML tags (`<br />`), URLs and extra whitespace (NLTK `WhitespaceTokenizer`).
* **Stop-words are NOT removed.** NLTK's stop-word list contains negations such as *not* and *no*; removing them would destroy information BERT needs (see the check below).
* Lower-casing is performed by the *uncased* BERT tokenizer.

In [ ]:
english_stopwords = set(stopwords.words("english"))
print("'not' in NLTK stop-words:", "not" in english_stopwords, "| 'no':", "no" in english_stopwords, "| 'but':", "but" in english_stopwords)

whitespace_tokenizer = WhitespaceTokenizer()

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", str(text))          # HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # URLs
    return " ".join(whitespace_tokenizer.tokenize(text))

df = pd.DataFrame({"review": raw_df["review"].apply(clean_text),
                   "label": (raw_df["sentiment"].str.lower() == "positive").astype(int)})
before = len(df)
df = df.drop_duplicates(subset="review").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate reviews; {len(df)} remain")

if NUM_SAMPLES and NUM_SAMPLES < len(df):
    df, _ = train_test_split(df, train_size=NUM_SAMPLES, stratify=df["label"], random_state=SEED)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print(f"Split (seed={SEED}, stratified): train={len(train_df)}  validation={len(val_df)}  test={len(test_df)}")
print("Train label balance:", train_df["label"].value_counts().to_dict())

df["label"].map(dict(enumerate(LABEL_NAMES))).value_counts().plot(kind="bar", title="Class balance", figsize=(4, 3))
plt.show()

## 3. BERT WordPiece tokenization – input IDs and attention masks

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

demo_text = "The movie was not bad, but unbelievably slow."
demo = tokenizer(demo_text, padding="max_length", max_length=16, truncation=True)
print("WordPieces    :", tokenizer.convert_ids_to_tokens(demo["input_ids"]))
print("input_ids     :", demo["input_ids"])
print("attention_mask:", demo["attention_mask"])

def make_loader(frame, shuffle):
    encoded = tokenizer(frame["review"].tolist(), padding="max_length", truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
    labels = torch.tensor(frame["label"].values, dtype=torch.long)
    return DataLoader(TensorDataset(encoded["input_ids"], encoded["attention_mask"], labels),
                      batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)
test_loader = make_loader(test_df, False)
print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

## 4. Fine-tune `bert-base-uncased` with a binary classification head

In [ ]:
set_seed()
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate(current_model, loader):
    current_model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = current_model(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE)).logits
            predictions.extend(logits.argmax(dim=-1).cpu().tolist())
            targets.extend(labels.tolist())
    return np.array(targets), np.array(predictions)

history = {"step_loss": [], "val_accuracy": []}
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, 1):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            loss = model(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE),
                         labels=labels.to(DEVICE)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step(); optimizer.zero_grad()
        running += loss.item(); history["step_loss"].append(loss.item())
        if step % 100 == 0:
            print(f"epoch {epoch + 1} step {step}/{len(train_loader)}  loss {running / step:.4f}")
    y_val, p_val = evaluate(model, val_loader)
    history["val_accuracy"].append(accuracy_score(y_val, p_val))
    print(f"==> Epoch {epoch + 1}: validation accuracy = {history['val_accuracy'][-1]:.4f}")

model.save_pretrained(SAVE_DIR); tokenizer.save_pretrained(SAVE_DIR)
plt.figure(figsize=(6, 3)); plt.plot(pd.Series(history["step_loss"]).rolling(50, min_periods=1).mean())
plt.xlabel("training step"); plt.ylabel("loss"); plt.title("Training loss (smoothed)"); plt.show()

## 5. Evaluation on the test set – accuracy, precision, recall, F1, confusion matrix

In [ ]:
y_test, p_test = evaluate(model, test_loader)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, p_test, average="binary")
bert_accuracy = accuracy_score(y_test, p_test)
print(f"Accuracy : {bert_accuracy:.4f}\nPrecision: {precision:.4f}\nRecall   : {recall:.4f}\nF1-score : {f1:.4f}\n")
print(classification_report(y_test, p_test, target_names=LABEL_NAMES, digits=4))

ConfusionMatrixDisplay(confusion_matrix(y_test, p_test), display_labels=LABEL_NAMES).plot(cmap="Blues")
plt.title("BERT – confusion matrix (test set)"); plt.show()

In [ ]:
# Bag-of-words baseline (TF-IDF + Logistic Regression) trained on the SAME training split
bow_vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 1), stop_words=None)
bow_train = bow_vectorizer.fit_transform(train_df["review"])
bow_model = LogisticRegression(max_iter=1000, random_state=SEED).fit(bow_train, train_df["label"])
bow_accuracy = accuracy_score(test_df["label"], bow_model.predict(bow_vectorizer.transform(test_df["review"])))
display(pd.DataFrame({"Model": ["TF-IDF + LogReg (bag-of-words)", "BERT (fine-tuned)"],
                      "Test accuracy": [round(bow_accuracy, 4), round(bert_accuracy, 4)]}))

## 6. Inference on unseen reviews – label + confidence
First the four sample reviews, then **12 additional reviews** (negation ×3, sarcasm ×2, mixed ×2, contrast ×2, plain ×2, ambiguous ×1).
The bag-of-words baseline is shown next to BERT so the "why a keyword classifier fails" argument is backed by real predictions.

In [ ]:
def predict(texts):
    """Return (label, confidence) for each text using the fine-tuned BERT model."""
    model.eval()
    encoded = tokenizer([clean_text(t) for t in texts], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        probabilities = torch.softmax(model(**encoded).logits.float(), dim=-1).cpu()
    confidence, index = probabilities.max(dim=-1)
    return [(LABEL_NAMES[i], round(c, 3)) for i, c in zip(index.tolist(), confidence.tolist())]

def bow_predict(texts):
    cleaned = [clean_text(t) for t in texts]
    return [LABEL_NAMES[i] for i in bow_model.predict(bow_vectorizer.transform(cleaned))]

SAMPLE_REVIEWS = [
    "The movie was an absolute masterpiece with brilliant performances and a gripping story.",
    "I found the plot boring and the acting was extremely weak throughout.",
    "A delightful film that kept me engaged from start to finish.",
    "This was one of the worst movies I have ever sat through.",
]
sample_preds = predict(SAMPLE_REVIEWS)
display(pd.DataFrame({"review": SAMPLE_REVIEWS, "BERT label": [p[0] for p in sample_preds],
                      "confidence": [p[1] for p in sample_preds], "BoW label": bow_predict(SAMPLE_REVIEWS)}))

In [ ]:
CUSTOM_REVIEWS = [
    # (review, category, intended sentiment, why a keyword/BoW classifier is likely to fail)
    ("The film was not bad at all; I actually enjoyed it.", "negation", "Positive",
     "'bad' is a negative keyword; the negation 'not ... at all' flips the meaning."),
    ("I did not hate this movie, it was surprisingly good.", "negation", "Positive",
     "'hate' counts as strongly negative in isolation."),
    ("It's hardly a boring movie; every scene kept me on the edge of my seat.", "negation", "Positive",
     "'boring' is negated by 'hardly'."),
    ("Oh great, another two hours of my life wasted on this masterpiece.", "sarcasm", "Negative",
     "'great' and 'masterpiece' are positive keywords but the intent is negative."),
    ("Wow, I loved how the plot made absolutely no sense whatsoever.", "sarcasm", "Negative",
     "'loved' is positive; the sarcasm is only visible from context."),
    ("The acting was superb, but the story was dull and the ending was a mess.", "mixed", "Negative",
     "Positive and negative words cancel in word counts; the model must weigh the clauses."),
    ("Beautiful cinematography, though the dialogue was painfully bad.", "mixed", "Negative",
     "Both polarities in one sentence; counts cannot say which dominates."),
    ("I expected a disaster but it turned out to be a pleasant surprise.", "contrast", "Positive",
     "'disaster' is a strong negative keyword but the sentence turns positive after 'but'."),
    ("Not the worst film ever made, but I would never watch it again.", "negation + contrast", "Negative",
     "'worst' + 'not' + 'but' interact; overall sentiment is negative."),
    ("A stunning, emotional journey that I will remember for years.", "plain positive", "Positive",
     "Easy case: both approaches should succeed."),
    ("Terrible pacing, wooden actors and a script full of cliches.", "plain negative", "Negative",
     "Easy case: both approaches should succeed."),
    ("The movie was fine, I guess.", "ambiguous", "Negative",
     "Weakly polarised; shows where BERT confidence is low."),
]
texts = [r[0] for r in CUSTOM_REVIEWS]
bert_out = predict(texts)
bow_out = bow_predict(texts)

custom_df = pd.DataFrame({
    "category": [r[1] for r in CUSTOM_REVIEWS],
    "review": texts,
    "intended": [r[2] for r in CUSTOM_REVIEWS],
    "BERT": [o[0] for o in bert_out],
    "confidence": [o[1] for o in bert_out],
    "BoW": bow_out,
    "why BoW struggles": [r[3] for r in CUSTOM_REVIEWS],
})
custom_df["BERT ok?"] = custom_df["BERT"] == custom_df["intended"]
custom_df["BoW ok?"] = custom_df["BoW"] == custom_df["intended"]
pd.set_option("display.max_colwidth", 90)
display(custom_df)
print(f"BERT matches the intended label on {custom_df['BERT ok?'].sum()}/{len(custom_df)}; "
      f"BoW on {custom_df['BoW ok?'].sum()}/{len(custom_df)}")

### Analysis (verify against the table; BERT can also fail)
* **Negation:** a bag-of-words model scores each word independently, so *"not bad"* inherits the weight of *bad*. BERT's contextual embeddings let *bad* be represented differently when preceded by *not*, so it usually succeeds.
* **Sarcasm:** positive words (*great, masterpiece, loved*) dominate keyword counts. BERT sometimes catches the sarcasm from cues such as "another two hours … wasted", but sarcasm is genuinely hard and **BERT may also fail** (look at the confidence column – low confidence indicates uncertainty).
* **Mixed opinions:** word counts ignore which clause carries the final judgement; BERT attends across the sentence and typically follows the clause after *but/though*.
* The ambiguous review ("fine, I guess") should have low confidence – an honest sign the model is unsure.

## 7. Bonus tasks
**(b) Attention visualisation** – one correctly and one incorrectly classified review (the incorrect one is taken automatically from the test set).
**(c) Interactive demo widget** (notebook version of the Streamlit/Flask demo).

In [ ]:
attention_model = BertForSequenceClassification.from_pretrained(SAVE_DIR, attn_implementation="eager").to(DEVICE).eval()

def plot_attention(text, title):
    encoded = tokenizer(clean_text(text), return_tensors="pt", truncation=True, max_length=48).to(DEVICE)
    with torch.no_grad():
        output = attention_model(**encoded, output_attentions=True)
    attention = output.attentions[-1][0].mean(dim=0).float().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.45), max(5, len(tokens) * 0.4)))
    image = ax.imshow(attention, cmap="viridis")
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
    ax.set_title(title); fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()

# correctly classified review
correct_text = SAMPLE_REVIEWS[1]
plot_attention(correct_text, f"Correct: {predict([correct_text])[0]}")

# a misclassified review from the test set
wrong_positions = np.where(p_test != y_test)[0]
if len(wrong_positions):
    wrong_index = int(wrong_positions[0])
    wrong_text = test_df.loc[wrong_index, "review"]
    print("Misclassified test review (true:", LABEL_NAMES[y_test[wrong_index]], "| predicted:", LABEL_NAMES[p_test[wrong_index]], ")")
    print(wrong_text[:300], "...")
    plot_attention(wrong_text, f"Incorrect: predicted {LABEL_NAMES[p_test[wrong_index]]}, true {LABEL_NAMES[y_test[wrong_index]]}")
else:
    print("No misclassified reviews in the test set.")

**(d) Comment (edit after viewing the heatmaps):** in the correct example, the [CLS] row (which feeds the classifier) and the tokens around *boring* / *weak* receive high attention, showing the
model focuses on the opinion words. In the misclassified review, attention is spread over many neutral tokens or concentrates on a misleading keyword, which explains the wrong prediction.
(Last-layer attention is only a rough explanation, not a full causal account.)

In [ ]:
import ipywidgets as widgets

text_box = widgets.Textarea(placeholder="Type a movie review", description="Review:",
                            layout=widgets.Layout(width="85%", height="90px"))
button = widgets.Button(description="Predict sentiment", button_style="primary")
output_area = widgets.Output()

def on_predict_clicked(_):
    with output_area:
        output_area.clear_output()
        if text_box.value.strip():
            label, confidence = predict([text_box.value])[0]
            print(f"{label}  (confidence {confidence:.3f})")

button.on_click(on_predict_clicked)
display(text_box, button, output_area)